In [27]:
import pandas as pd

### Task 0: Validate and Load the Dataset
- Use the validation code you created in previous units to confirm that all required columns exist

- Convert the order_date column to datetime format using pd.to_datetime()

- Display the first 5 rows

In [28]:
# Load the master data from a CSV file
df_master_data = pd.read_csv("master_data.csv")

# Check if the columns exists and throw an error if not: order_id, customer_id, customer_name, product_id, product_name, category, quantity, price, region, order_date
required_columns = ["order_id", "customer_id", "customer_name", "product_id", "product_name", "category", "quantity", "price", "region", "order_date"]
for column in required_columns:    
    if column not in df_master_data.columns:
        raise ValueError(f"Column '{column}' is missing from the master data.") 
    
# Convert the order_date column to datetime format
df_master_data["order_date"] = pd.to_datetime(df_master_data["order_date"])    
# Display the first five rows of the master data
print(df_master_data.head(5))

   order_id  customer_id  product_id  quantity order_date product_name  \
0      3001         1048        2020         4 2025-04-03       Second   
1      3002         1015        2011         4 2025-04-20       Ground   
2      3003         1008        2007         2 2025-04-04       Camera   
3      3004         1014        2001         4 2025-04-01         Rate   
4      3005         1023        2001         4 2025-04-30         Rate   

   category   price     customer_name region  
0  Clothing  302.97   Shane Henderson   East  
1     Books  334.64      Dylan Miller   West  
2      Home  252.64        Gina Moore  North  
3     Books  406.11  Michele Williams   East  
4     Books  406.11       Ethan Adams  South  


### Task 1: Time-Based Slicing and Filtering
Set order_date as the DataFrame index

Extract and display:

- All orders from March 2023

- All weekday orders (ignore Saturday and Sunday)

In [29]:
# Set order_date as the index of the DataFrame
if df_master_data.index.name != "order_date":
    df_master_data.set_index("order_date", inplace=True)

In [30]:
# I'm displaying all orders from April 2025 because there is not March 2023 data. 
# Index is in datetime format of YYYY-MM-DD
# To display March of 2023, use df_master_data.loc["2023-03"]
df_april_2025 = df_master_data.loc["2025-04"]
print(df_april_2025)

            order_id  customer_id  product_id  quantity product_name  \
order_date                                                             
2025-04-03      3001         1048        2020         4       Second   
2025-04-20      3002         1015        2011         4       Ground   
2025-04-04      3003         1008        2007         2       Camera   
2025-04-01      3004         1014        2001         4         Rate   
2025-04-30      3005         1023        2001         4         Rate   
2025-04-04      3006         1040        2020         2       Second   
2025-04-21      3008         1016        2009         4         Down   
2025-04-25      3009         1045        2003         2         Left   
2025-04-18      3011         1047        2006         3         Live   
2025-04-10      3013         1026        2009         1         Down   
2025-04-21      3017         1029        2010         2       Attack   
2025-04-16      3020         1001        2009         3         

In [31]:
# Display all weekday orders except Saturdays and Sundays
df_weekday_orders = df_master_data[df_master_data.index.dayofweek < 5]      
print(df_weekday_orders)

            order_id  customer_id  product_id  quantity product_name  \
order_date                                                             
2025-04-03      3001         1048        2020         4       Second   
2025-04-04      3003         1008        2007         2       Camera   
2025-04-01      3004         1014        2001         4         Rate   
2025-04-30      3005         1023        2001         4         Rate   
2025-04-04      3006         1040        2020         2       Second   
...              ...          ...         ...       ...          ...   
2025-04-04      3093         1002        2016         3          Per   
2025-05-07      3095         1045        2003         1         Left   
2025-04-09      3096         1006        2020         4       Second   
2025-05-23      3097         1028        2001         3         Rate   
2025-04-21      3098         1028        2020         1       Second   

            category   price     customer_name region  
order_d

### Task 2: Create a Rolling Revenue Trend
- Compute a new column daily_revenue = quantity * price

- Group by order_date and sum daily_revenue

- Apply a 7-day rolling average to the daily revenue totals

- Display the first 15 rows of the result

In [32]:
# Compute daily_revenue by multiplying quantity and price, then resampling by day and summing the revenue for each day
df_master_data["daily_revenue"] = df_master_data["quantity"] * df_master_data["price"]

# Group by order_date and sum dailey_revenue, then apply a 7-day rolling average to the daily_revenue column
daily_revenue = df_master_data.groupby("order_date")["daily_revenue"].sum()
daily_revenue_rolling = daily_revenue.rolling(window=7).mean()  

# Display the first 15 rows
print(daily_revenue_rolling.head(15))

order_date
2025-03-29            NaN
2025-03-30            NaN
2025-04-01            NaN
2025-04-02            NaN
2025-04-03            NaN
2025-04-04            NaN
2025-04-05    2006.845714
2025-04-07    2329.542857
2025-04-08    2504.491429
2025-04-09    2445.554286
2025-04-10    2421.851429
2025-04-11    1711.052857
2025-04-13    1817.237143
2025-04-14    1817.152857
2025-04-15    1331.181429
Name: daily_revenue, dtype: float64


### Task 3: Weekly Resampling with resample() and Export
Group data by week using resample('W') on order_date

For each week, calculate:

- Total revenue

- Average order value

Save the resulting DataFrame as weekly_trends.csv

In [33]:
# Group data by week using resample('W') on order_date and calculate total revenue and average order value for each week
weekly_summary = df_master_data.resample("W")["daily_revenue"].agg(
    total_revenue="sum",
    avg_order_value="mean"
)

# Save the results as weekly_trends.csv
weekly_summary.to_csv("weekly_trends.csv", index=True)